# Drug Policy Criminalization Analysis

This notebook covers all analysis for comparing drug enforcement vs. treatment orientation across U.S. states (2015–2022).

## Instructions
- Run each cell in order
- Modify the configuration parameters in the cell below for different experiments
- Outputs are saved to the `outputs/` directory

---
## Imports and Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import us

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import statsmodels.api as sm
from xgboost import XGBRegressor
import time

warnings.filterwarnings("ignore")

BASE    = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..")
DATA    = os.path.join(BASE, "data/processed/panel_dataset.csv")
OUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

---
## Data Loading (PART 1)

Loads the panel dataset built from NIBRS drug arrests, SAMHSA treatment admissions,
Census ACS demographics, and policy/structural covariates (2015–2022, 49 states).

In [ ]:
df = pd.read_csv(DATA)

print(f"Dataset loaded: {len(df)} state-year observations")
print(f"States: {df['state'].nunique()}  |  Years: {sorted(df['year'].unique())}")
print(f"Columns: {list(df.columns)}")

---
Configuration Parameters - (to be modified)

In [ ]:
# ===== MODIFY THESE PARAMETERS FOR YOUR EXPERIMENTS =====

DROP_OUTLIERS  = True          # PART 2: Filter known data-quality outliers
FEATURES_SET   = "extended"    # PART 4: "baseline" or "extended" (with interactions & policy)
LOG_TRANSFORM  = True          # PART 4: Log-transform the target variable
N_ESTIMATORS   = 400           # PART 5: Number of trees in RF / GB models
RUN_TUNING     = True          # PART 6: Run RandomizedSearchCV (slower but better)
BEST_MODEL     = "auto"        # PART 7: "auto" selects best from Part 6, or set "rf"/"gb"/"xgb"

# PART 4: Available Feature Sets
FEATURE_SETS = {
    "baseline": [
        'poverty_rate', 'median_income', 'unemployment_rate',
        'pct_white', 'pct_black', 'pct_hispanic', 'year',
    ],
    "extended": [
        'poverty_rate', 'median_income', 'unemployment_rate',
        'pct_white', 'pct_black', 'pct_hispanic',
        'overdose_death_rate', 'incarceration_rate',
        'marijuana_legal', 'republican_gov',
        'pres_vote_rep', 'gov_streak',
        'senate_rep_pct', 'house_rep_pct',
        'align_unified_dem', 'align_div_rep_gov', 'align_div_dem_gov',
        'year',
    ],
}

# =========================================================

---
## Data Quality Filtering (PART 2)

Removes state-years with known data-quality issues:
- **Idaho 2022**: Missing NIBRS data
- **Florida 2017–2021**: Incomplete ASR reporting during NIBRS transition
- **Illinois 2020–2021**: NIBRS transition artifacts

Set `DROP_OUTLIERS = False` to include all observations and compare.

In [ ]:
if DROP_OUTLIERS:
    bad_mask = (
        ((df['state'] == 'Idaho')    & (df['year'] == 2022)) |
        ((df['state'] == 'Florida')  & (df['year'].isin([2017, 2018, 2019, 2020, 2021]))) |
        ((df['state'] == 'Illinois') & (df['year'].isin([2020, 2021])))
    )
    clean = df[~bad_mask].copy()
else:
    clean = df.copy()

complete = clean.dropna(subset=['criminalization_index', 'arrest_rate',
                                 'treatment_rate', 'poverty_rate', 'population'])

print("=" * 60)
print("DATA CONFIGURATION")
print("=" * 60)
print(f"Drop Outliers:      {DROP_OUTLIERS}")
print(f"Total rows:         {len(df)}")
print(f"After filtering:    {len(clean)}")
print(f"Complete obs:       {len(complete)}")
print(f"States:             {complete['state'].nunique()}")
print(f"Years:              {sorted(complete['year'].unique())}")
print("=" * 60)

print("\nCriminalization Index summary:")
print(complete['criminalization_index'].describe().round(3).to_string())

---
## Exploratory Data Analysis (PART 3)

Produces three visualizations:
1. **Choropleth map** — average Criminalization Index by state
2. **Time trends** — national arrest rate vs. treatment admission rate (2015–2022)
3. **Distribution + correlation heatmap** — index distribution and feature correlations

Toggle `DROP_OUTLIERS` in the config to see how outlier removal affects the plots.

In [ ]:
# --- Fig 1: Choropleth -------------------------------------------------------
state_avg = (complete.groupby('state')['criminalization_index']
             .mean().reset_index()
             .rename(columns={'criminalization_index': 'avg_index'}))

state_avg['code'] = state_avg['state'].apply(
    lambda s: us.states.lookup(s).abbr if us.states.lookup(s) else None)
state_avg = state_avg.dropna(subset=['code'])

fig1 = px.choropleth(
    state_avg,
    locations='code',
    locationmode='USA-states',
    color='avg_index',
    scope='usa',
    color_continuous_scale='RdYlGn_r',
    labels={'avg_index': 'Avg Criminalization Index'},
    title='Average Criminalization Index by State (2015\u20132022)<br>'
          '<sup>Higher = more arrests relative to treatment admissions</sup>',
)
fig1.update_layout(
    coloraxis_colorbar=dict(title='Index'),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig1.write_html(os.path.join(OUT_DIR, "fig1_choropleth.html"))
fig1.show()
print("Saved: fig1_choropleth.html")

# --- Fig 2: Time trends -------------------------------------------------------
national = (complete.groupby('year')
            .apply(lambda g: pd.Series({
                'arrest_rate':    np.average(g['arrest_rate'],    weights=g['population']),
                'treatment_rate': np.average(g['treatment_rate'], weights=g['population']),
            }), include_groups=False)
            .reset_index())

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=national['year'], y=national['arrest_rate'],
    mode='lines+markers', name='Drug Arrest Rate',
    line=dict(color='#d62728', width=2.5), marker=dict(size=7),
))
fig2.add_trace(go.Scatter(
    x=national['year'], y=national['treatment_rate'],
    mode='lines+markers', name='Treatment Admission Rate',
    line=dict(color='#1f77b4', width=2.5), marker=dict(size=7),
))
fig2.update_layout(
    title='National Drug Arrest Rate vs. Treatment Admission Rate (per 100k)<br>'
          '<sup>Population-weighted average across 49 states, 2015\u20132022</sup>',
    xaxis_title='Year', yaxis_title='Rate per 100,000',
    legend=dict(x=0.01, y=0.99), hovermode='x unified',
)
fig2.write_html(os.path.join(OUT_DIR, "fig2_time_trends.html"))
fig2.show()
print("Saved: fig2_time_trends.html")

# --- Fig 3: Distribution + correlation heatmap --------------------------------
matplotlib.use("Agg")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(complete['criminalization_index'], bins=40,
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(complete['criminalization_index'].median(), color='red',
                linestyle='--',
                label=f"Median = {complete['criminalization_index'].median():.2f}")
axes[0].axvline(1.0, color='black', linestyle=':', linewidth=1.2, label='Index = 1.0')
axes[0].set_xlabel('Criminalization Index')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Criminalization Index')
axes[0].legend()

corr_cols = ['criminalization_index', 'arrest_rate', 'treatment_rate',
             'poverty_rate', 'median_income', 'unemployment_rate',
             'pct_white', 'pct_black', 'pct_hispanic']
corr = complete[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1],
            annot_kws={'size': 8},
            xticklabels=[c.replace('_', ' ') for c in corr_cols],
            yticklabels=[c.replace('_', ' ') for c in corr_cols])
axes[1].set_title('Correlation Matrix')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_distributions.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_distributions.png")

---
## Feature Engineering (PART 4)

Builds the modeling feature matrix:
- Selects features from `FEATURES_SET` (baseline or extended)
- Adds **region dummies** (Northeast, Midwest, West; South = reference)
- Adds **interaction terms** (e.g., presidential vote × overdose rate)
- Log-transforms the target if `LOG_TRANSFORM = True`

Switch `FEATURES_SET = "baseline"` to compare a minimal vs. full feature set.

In [ ]:
FEATURES = FEATURE_SETS[FEATURES_SET]
TARGET   = 'criminalization_index'

model_df = complete.dropna(subset=FEATURES + [TARGET]).copy()

if LOG_TRANSFORM:
    model_df['log_index'] = np.log(model_df[TARGET])
    y_col = 'log_index'
else:
    y_col = TARGET

# Region dummies (ref = South)
NORTHEAST = {'Connecticut','Maine','Maryland','Massachusetts','New Hampshire',
             'New Jersey','New York','Pennsylvania','Rhode Island','Vermont','Delaware'}
MIDWEST   = {'Illinois','Indiana','Iowa','Kansas','Michigan','Minnesota',
             'Missouri','Nebraska','North Dakota','Ohio','South Dakota','Wisconsin'}
WEST      = {'Alaska','Arizona','California','Colorado','Hawaii','Idaho',
             'Montana','Nevada','New Mexico','Utah','Washington','Wyoming'}
model_df['region_northeast'] = model_df['state'].isin(NORTHEAST).astype(float)
model_df['region_midwest']   = model_df['state'].isin(MIDWEST).astype(float)
model_df['region_west']      = model_df['state'].isin(WEST).astype(float)

# Interaction terms (only for extended feature set)
if FEATURES_SET == "extended":
    model_df['pres_x_overdose']         = model_df['pres_vote_rep'] * model_df['overdose_death_rate']
    model_df['pres_x_incarceration']    = model_df['pres_vote_rep'] * model_df['incarceration_rate']
    model_df['rep_pct_x_incarceration'] = model_df['senate_rep_pct'] * model_df['incarceration_rate']
    model_df['incarc_x_poverty']        = model_df['incarceration_rate'] * model_df['poverty_rate']
    model_df['overdose_x_mj']           = model_df['overdose_death_rate'] * model_df['marijuana_legal']
    model_df['incarc_x_midwest']        = model_df['incarceration_rate'] * model_df['region_midwest']
    model_df['incarc_x_south']          = (
        model_df['incarceration_rate'] *
        (1 - model_df['region_northeast'] - model_df['region_midwest'] - model_df['region_west'])
    )
    FEATURES_ALL = FEATURES + [
        'region_northeast', 'region_midwest', 'region_west',
        'pres_x_overdose', 'pres_x_incarceration', 'rep_pct_x_incarceration',
        'incarc_x_poverty', 'overdose_x_mj', 'incarc_x_midwest', 'incarc_x_south',
    ]
else:
    FEATURES_ALL = FEATURES + ['region_northeast', 'region_midwest', 'region_west']

X = model_df[FEATURES_ALL].values
y = model_df[y_col].values

print(f"Feature set:   {FEATURES_SET} -> {len(FEATURES_ALL)} features")
print(f"Observations:  {len(model_df)}")
target_label = "log(Criminalization Index)" if LOG_TRANSFORM else "Criminalization Index"
print(f"Target:        {target_label}  |  mean={y.mean():.3f}  std={y.std():.3f}")

---
## Model Training and Comparison (PART 5)

Trains five models with 5-fold cross-validation:
1. **OLS** — linear baseline with standardized features
2. **Random Forest** — 400 trees, max depth 6
3. **Extra Trees** — 400 trees, max depth 6
4. **Gradient Boosting** — 500 estimators
5. **XGBoost** — 500 estimators

Adjust `N_ESTIMATORS` in the config to trade accuracy for speed.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)
X_sm      = sm.add_constant(X_scaled)
ols_model = sm.OLS(y, X_sm).fit()

ols_sk = LinearRegression()
rf     = RandomForestRegressor(n_estimators=N_ESTIMATORS, max_depth=6,
                                min_samples_leaf=3, random_state=42, n_jobs=-1)
et     = ExtraTreesRegressor(n_estimators=N_ESTIMATORS, max_depth=6,
                              min_samples_leaf=3, random_state=42, n_jobs=-1)
gb     = GradientBoostingRegressor(n_estimators=N_ESTIMATORS+100, max_depth=4,
                                    learning_rate=0.04, subsample=0.7,
                                    min_samples_leaf=4, random_state=42)
xgb    = XGBRegressor(n_estimators=N_ESTIMATORS+100, max_depth=4, learning_rate=0.04,
                       subsample=0.7, colsample_bytree=0.8, min_child_weight=4,
                       random_state=42, n_jobs=-1, verbosity=0)

def cv_metrics(model, Xm, ym, label):
    r2s   = cross_val_score(model, Xm, ym, cv=kf, scoring='r2')
    maes  = -cross_val_score(model, Xm, ym, cv=kf, scoring='neg_mean_absolute_error')
    rmses = np.sqrt(-cross_val_score(model, Xm, ym, cv=kf, scoring='neg_mean_squared_error'))
    return {'label': label, 'R2': r2s.mean(), 'R2_std': r2s.std(),
            'MAE': maes.mean(), 'RMSE': rmses.mean()}

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"Feature Set:        {FEATURES_SET} -> {len(FEATURES_ALL)} features")
print(f"N Estimators:       {N_ESTIMATORS}")
print(f"Log Transform:      {LOG_TRANSFORM}")
print(f"Observations:       {len(model_df)}")
print(f"CV Folds:           5")
print("=" * 60)

start_time = time.time()

results = []
for model, Xm, label in [
    (ols_sk, X_scaled, "OLS Linear Regression"),
    (rf,     X,        "Random Forest"),
    (et,     X,        "Extra Trees"),
    (gb,     X,        "Gradient Boosting"),
    (xgb,    X,        "XGBoost"),
]:
    results.append(cv_metrics(model, Xm, y, label))
    print(f"  {label}: R\u00b2={results[-1]['R2']:.3f}")

training_time = time.time() - start_time

print(f"\nTraining Time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

res_df = pd.DataFrame(results)[['label', 'R2', 'R2_std', 'MAE', 'RMSE']]

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(res_df.to_string(index=False))
print("=" * 60)

---
## Hyperparameter Tuning (PART 6)

Runs `RandomizedSearchCV` (60 iterations, 5-fold CV) to find the best configuration for:
- **XGBoost**: n_estimators, max_depth, learning_rate, subsample, colsample_bytree
- **Gradient Boosting**: n_estimators, max_depth, learning_rate, subsample, max_features

Set `RUN_TUNING = False` to skip this step and use default parameters.

**Expected observations:**
- Tuned models typically improve R² by 0.02–0.08 over defaults
- XGBoost and GB usually perform similarly after tuning
- Best R² on this dataset is typically 0.55–0.75

In [ ]:
if RUN_TUNING:
    xgb_param_dist = {
        'n_estimators':     [300, 400, 500, 700],
        'max_depth':        [3, 4, 5, 6],
        'learning_rate':    [0.02, 0.03, 0.05, 0.07, 0.1],
        'subsample':        [0.6, 0.7, 0.8],
        'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
        'min_child_weight': [3, 4, 5, 6],
        'reg_alpha':        [0, 0.01, 0.1, 0.5],
        'reg_lambda':       [0.5, 1, 2, 5],
    }
    gb_param_dist = {
        'n_estimators':     [300, 500, 700, 1000],
        'max_depth':        [3, 4, 5],
        'learning_rate':    [0.02, 0.03, 0.05, 0.07],
        'subsample':        [0.6, 0.7, 0.8],
        'min_samples_leaf': [3, 4, 5, 6, 8],
        'max_features':     [0.6, 0.7, 0.8, 'sqrt'],
    }

    print("Running RandomizedSearchCV for XGBoost...")
    tune_start = time.time()
    xgb_search = RandomizedSearchCV(
        XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
        param_distributions=xgb_param_dist,
        n_iter=60, cv=kf, scoring='r2', random_state=42, n_jobs=1,
    )
    xgb_search.fit(X, y)
    xgb_tuned_r2 = xgb_search.best_score_
    print(f"  Best XGBoost R\u00b2 (CV): {xgb_tuned_r2:.3f}")
    print(f"  Best params: {xgb_search.best_params_}")

    print("\nRunning RandomizedSearchCV for Gradient Boosting...")
    gb_search = RandomizedSearchCV(
        GradientBoostingRegressor(random_state=42),
        param_distributions=gb_param_dist,
        n_iter=60, cv=kf, scoring='r2', random_state=42, n_jobs=-1,
    )
    gb_search.fit(X, y)
    gb_tuned_r2 = gb_search.best_score_
    print(f"  Best GB R\u00b2     (CV): {gb_tuned_r2:.3f}")
    print(f"  Best params: {gb_search.best_params_}")

    tuning_time = time.time() - tune_start
    best_r2    = max(xgb_tuned_r2, gb_tuned_r2)
    best_label = 'XGBoost (tuned)' if xgb_tuned_r2 >= gb_tuned_r2 else 'Gradient Boosting (tuned)'
    best_model_obj = xgb_search.best_estimator_ if xgb_tuned_r2 >= gb_tuned_r2 else gb_search.best_estimator_

    print(f"\nTuning Time: {tuning_time:.2f} seconds ({tuning_time/60:.2f} minutes)")
    print("\n" + "=" * 60)
    print("TUNING RESULTS SUMMARY")
    print("=" * 60)
    print(f"Best model: {best_label}")
    print(f"Best R\u00b2:    {best_r2:.3f}")
    print("=" * 60)
else:
    # Fit defaults
    xgb.fit(X, y); gb.fit(X, y)
    best_model_obj = xgb
    best_label     = "XGBoost (default)"
    best_r2        = results[4]['R2']
    print(f"Tuning skipped. Using {best_label}  R\u00b2={best_r2:.3f}")

---
## Final Model and Visualizations (PART 7)

Generates five publication-ready visualizations using the best model from Part 6:

1. **Choropleth** — drug policy orientation by state
2. **Time trends** — dual-axis national trends with criminalization index
3. **Feature importance** — interactive horizontal bar chart
4. **Scatter + regression** — top predictor vs. criminalization index
5. **State rankings** — all 49 states ranked by average index

Set `BEST_MODEL = "rf"`, `"gb"`, or `"xgb"` to override the auto-selected model.

In [ ]:
# Select final model
if BEST_MODEL == "auto":
    final_model = best_model_obj
    final_label = best_label
elif BEST_MODEL == "rf":
    rf.fit(X, y); final_model = rf; final_label = "Random Forest"
elif BEST_MODEL == "gb":
    gb.fit(X, y); final_model = gb; final_label = "Gradient Boosting"
else:
    xgb.fit(X, y); final_model = xgb; final_label = "XGBoost"

final_model.fit(X, y)

feat_labels_map = {
    'poverty_rate': 'Poverty Rate', 'median_income': 'Median Income',
    'unemployment_rate': 'Unemployment Rate', 'pct_white': '% White',
    'pct_black': '% Black', 'pct_hispanic': '% Hispanic', 'year': 'Year',
    'overdose_death_rate': 'Overdose Death Rate', 'incarceration_rate': 'Incarceration Rate',
    'marijuana_legal': 'Marijuana Legal', 'republican_gov': 'Republican Governor',
    'pres_vote_rep': 'Presidential Vote (Rep)', 'gov_streak': 'Governor Streak',
    'senate_rep_pct': 'Senate Rep %', 'house_rep_pct': 'House Rep %',
    'align_unified_dem': 'Unified Dem', 'align_div_rep_gov': 'Div (Rep Gov)',
    'align_div_dem_gov': 'Div (Dem Gov)',
    'region_northeast': 'Northeast', 'region_midwest': 'Midwest', 'region_west': 'West',
    'pres_x_overdose': 'PV \u00d7 Overdose', 'pres_x_incarceration': 'PV \u00d7 Incarceration',
    'rep_pct_x_incarceration': 'Rep% \u00d7 Incarceration', 'incarc_x_poverty': 'Incarc \u00d7 Poverty',
    'overdose_x_mj': 'Overdose \u00d7 MJ Legal', 'incarc_x_midwest': 'Incarc \u00d7 Midwest',
    'incarc_x_south': 'Incarc \u00d7 South',
}
feat_display = [feat_labels_map.get(f, f) for f in FEATURES_ALL]

# \u2500\u2500 State averages for maps/rankings \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
state_avg_full = (complete.groupby('state')
                  .agg(avg_index=('criminalization_index','mean'),
                       avg_arrest=('arrest_rate','mean'),
                       avg_treatment=('treatment_rate','mean'))
                  .reset_index())
state_avg_full['code'] = state_avg_full['state'].apply(
    lambda s: us.states.lookup(s).abbr if us.states.lookup(s) else None)
state_avg_full = state_avg_full.dropna(subset=['code'])
state_avg_full['orientation'] = state_avg_full['avg_index'].apply(
    lambda x: 'Criminalization-oriented' if x >= 1 else 'Treatment-oriented')

# VIZ 1: Choropleth -----------------------------------------------------------
viz1 = px.choropleth(
    state_avg_full, locations='code', locationmode='USA-states',
    color='avg_index', scope='usa', color_continuous_scale='RdYlGn_r',
    hover_name='state',
    hover_data={'code': False, 'avg_index': ':.2f',
                'avg_arrest': ':.1f', 'avg_treatment': ':.1f'},
    labels={'avg_index': 'Criminalization Index',
            'avg_arrest': 'Avg Arrest Rate/100k',
            'avg_treatment': 'Avg Treatment Rate/100k'},
    title='<b>Drug Policy Orientation by State</b><br>'
          '<sup>Average Criminalization Index 2015\u20132022 | '
          'Higher = more arrests relative to treatment admissions</sup>',
)
viz1.update_layout(coloraxis_colorbar=dict(title='Index', tickformat='.1f'),
                   margin=dict(l=0, r=0, t=70, b=0),
                   font=dict(family='Arial', size=12))
viz1.write_html(os.path.join(OUT_DIR, "viz1_choropleth_final.html"))
viz1.show()
print("Saved: viz1_choropleth_final.html")

# VIZ 2: Time trends ----------------------------------------------------------
national_full = (complete.groupby('year')
                 .apply(lambda g: pd.Series({
                     'arrest_rate':    np.average(g['arrest_rate'],    weights=g['population']),
                     'treatment_rate': np.average(g['treatment_rate'], weights=g['population']),
                     'avg_index':      np.average(g['criminalization_index'], weights=g['population']),
                 }), include_groups=False)
                 .reset_index())

viz2 = make_subplots(specs=[[{"secondary_y": True}]])
viz2.add_trace(go.Scatter(x=national_full['year'], y=national_full['arrest_rate'],
    name='Drug Arrest Rate', mode='lines+markers',
    line=dict(color='#d62728', width=2.5), marker=dict(size=8)), secondary_y=False)
viz2.add_trace(go.Scatter(x=national_full['year'], y=national_full['treatment_rate'],
    name='Treatment Admission Rate', mode='lines+markers',
    line=dict(color='#1f77b4', width=2.5), marker=dict(size=8)), secondary_y=False)
viz2.add_trace(go.Scatter(x=national_full['year'], y=national_full['avg_index'],
    name='Criminalization Index', mode='lines+markers',
    line=dict(color='#2ca02c', width=2, dash='dot'), marker=dict(size=7)), secondary_y=True)
viz2.add_hline(y=1.0, secondary_y=True,
               line=dict(color='gray', dash='dash', width=1),
               annotation_text='Index = 1.0', annotation_position='right')
viz2.update_layout(
    title='<b>National Drug Arrest vs. Treatment Rates Over Time</b><br>'
          '<sup>Population-weighted average, 49 states, 2015\u20132022</sup>',
    hovermode='x unified', legend=dict(x=0.01, y=0.99),
    font=dict(family='Arial', size=12))
viz2.update_yaxes(title_text='Rate per 100,000', secondary_y=False)
viz2.update_yaxes(title_text='Criminalization Index', secondary_y=True)
viz2.write_html(os.path.join(OUT_DIR, "viz2_time_trends_final.html"))
viz2.show()
print("Saved: viz2_time_trends_final.html")

# VIZ 3: Feature importance ---------------------------------------------------
imp_df = pd.DataFrame({'Feature': feat_display,
                        'Importance': final_model.feature_importances_}).sort_values('Importance')

viz3 = px.bar(imp_df, x='Importance', y='Feature', orientation='h',
              color='Importance', color_continuous_scale='Blues',
              title=f'<b>What Predicts Criminalization Index?</b><br>'
                    f'<sup>{final_label} feature importance (higher = stronger predictor)</sup>',
              labels={'Importance': 'Feature Importance', 'Feature': ''})
viz3.update_layout(coloraxis_showscale=False, font=dict(family='Arial', size=12),
                   margin=dict(l=150, r=40, t=80, b=40))
viz3.write_html(os.path.join(OUT_DIR, "viz3_feature_importance.html"))
viz3.show()
print("Saved: viz3_feature_importance.html")

# VIZ 4: Top predictor scatter ------------------------------------------------
top_feat_idx  = np.argmax(final_model.feature_importances_)
top_feat      = FEATURES_ALL[top_feat_idx]
top_label_str = feat_display[top_feat_idx]

scatter_df = complete.dropna(subset=[top_feat, 'criminalization_index']).copy()

viz4 = px.scatter(scatter_df, x=top_feat, y='criminalization_index',
                  color='state', hover_name='state', log_y=True, trendline='ols',
                  hover_data={'year': True, 'arrest_rate': ':.1f',
                              'treatment_rate': ':.1f', 'state': False},
                  labels={top_feat: top_label_str,
                          'criminalization_index': 'Criminalization Index (log scale)'},
                  title=f'<b>Criminalization Index vs. {top_label_str}</b><br>'
                        '<sup>Each point = one state-year | log scale | OLS trend line</sup>')
viz4.update_traces(marker=dict(size=6, opacity=0.6))
viz4.update_layout(showlegend=False, font=dict(family='Arial', size=12))
viz4.write_html(os.path.join(OUT_DIR, "viz4_scatter_final.html"))
viz4.show()
print("Saved: viz4_scatter_final.html")

# VIZ 5: State rankings -------------------------------------------------------
ranked = state_avg_full.sort_values('avg_index', ascending=True)
ranked['color'] = ranked['avg_index'].apply(lambda x: '#d62728' if x >= 1 else '#1f77b4')

viz5 = go.Figure(go.Bar(
    x=ranked['avg_index'], y=ranked['state'], orientation='h',
    marker_color=ranked['color'],
    text=ranked['avg_index'].round(2), textposition='outside',
    hovertemplate='<b>%{y}</b><br>Index: %{x:.2f}<extra></extra>',
))
viz5.add_vline(x=1.0, line=dict(color='black', dash='dash', width=1.5),
               annotation_text='Index = 1.0', annotation_position='top right')
viz5.update_layout(
    title='<b>Average Criminalization Index by State (2015\u20132022)</b><br>'
          '<sup>Red = more arrests than treatment | Blue = more treatment than arrests</sup>',
    xaxis_title='Criminalization Index', yaxis_title='',
    height=1100, margin=dict(l=150, r=80, t=80, b=40),
    font=dict(family='Arial', size=11))
viz5.write_html(os.path.join(OUT_DIR, "viz5_state_rankings.html"))
viz5.show()
print("Saved: viz5_state_rankings.html")

# Predicted vs Actual ---------------------------------------------------------
matplotlib.use("Agg")
preds = final_model.predict(X)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y, preds, alpha=0.5, color='steelblue', s=25, edgecolors='none')
lims = [min(y.min(), preds.min()) - 0.2, max(y.max(), preds.max()) + 0.2]
axes[0].plot(lims, lims, 'r--', linewidth=1.2, label='Perfect fit')
axes[0].set_xlabel(f'Actual {"log(Criminalization Index)" if LOG_TRANSFORM else "Criminalization Index"}')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'{final_label}: Predicted vs Actual')
axes[0].text(0.05, 0.92, f'R\u00b2 = {r2_score(y, preds):.3f}', transform=axes[0].transAxes)
axes[0].legend()

x_top = model_df[top_feat].values
m, b  = np.polyfit(x_top, y, 1)
x_line = np.linspace(x_top.min(), x_top.max(), 100)
axes[1].scatter(x_top, y, alpha=0.45, color='darkorange', s=25, edgecolors='none')
axes[1].plot(x_line, m * x_line + b, 'r-', linewidth=2)
axes[1].set_xlabel(top_label_str)
axes[1].set_ylabel(f'{"log(" if LOG_TRANSFORM else ""}Criminalization Index{")" if LOG_TRANSFORM else ""}')
axes[1].set_title(f'Top Predictor: {top_label_str}')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig3_feature_importance.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig3_feature_importance.png")

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"Final Model:         {final_label}")
print(f"In-sample R\u00b2:        {r2_score(y, preds):.3f}")
print(f"Top Predictor:       {top_label_str}")
print(f"Outputs saved to:    {OUT_DIR}")
print("=" * 60)

---
## Quick Experiment Guide

### PART 1: Baseline
- Run all cells with default settings
- Record the model comparison R\u00b2 table and training time

### PART 2: Data Quality Filtering
- Change `DROP_OUTLIERS = False` and re-run from the filtering cell
- Compare index distribution and model R\u00b2 with and without outliers
- **Why filtering matters:** Idaho 2022, Florida 2017\u20132021, and Illinois 2020\u20132021 have
  known NIBRS transition artifacts that inflate or suppress arrest counts

### PART 3: EDA
- Toggle `DROP_OUTLIERS` to see how data quality affects the choropleth and trend lines
- Examine the correlation heatmap to identify which demographics correlate most with the index
- Use the EDA outputs to form hypotheses about which features should matter most

### PART 4: Feature Engineering
- Change `FEATURES_SET = "baseline"` to use only 7 demographic features
- Compare R\u00b2 against the extended feature set (which adds policy and structural variables)
- Toggle `LOG_TRANSFORM = False` to model the raw index and observe skew effects

### PART 5: Model Comparison
- Change `N_ESTIMATORS` to 100 for faster training; 800+ for highest accuracy
- Note which model achieves the best cross-validated R\u00b2
- Examine the OLS coefficients for interpretability

### PART 6: Hyperparameter Tuning
- Set `RUN_TUNING = False` to skip and use default parameters
- Compare tuned R\u00b2 vs. default R\u00b2 from Part 5
- Read the best_params output to understand which hyperparameters matter most

### PART 7: Final Model
- Set `BEST_MODEL = "auto"` to use the best-tuned model from Part 6
- Or manually set `BEST_MODEL = "rf"` / `"gb"` / `"xgb"` to compare final maps
- Combine your best settings from all experiments for the final report

---

**Tips:**
- Always re-run the filtering cell (Part 2) after changing `DROP_OUTLIERS`
- Always re-run the feature engineering cell (Part 4) after changing `FEATURES_SET`
- Take screenshots of the Plotly maps and charts for your report
- Keep notes of each experiment's R\u00b2 and top predictor